# ScentAI — Offline Pipeline (One-Time)

This notebook consolidates all one-time offline processing for the ScentAI project.
It is designed to run on **Google Colab with an A100 (80 GB) or Blackwell 6000 GPU**.

## What this does
1. **LLM Enrichment** — fills all empty enrichment columns (`vibe_sentence`, `formality`, `day_night`, etc.)
   for all ~35k rows in `data/vibescent_enriched.csv` using Qwen3-8B (local GPU, no API keys).
2. **Corpus Re-embedding** — embeds every row's `retrieval_text` with `Qwen3-VL-Embedding-8B`
   and saves the resulting matrix to `artifacts/qwen3vl_corpus/embeddings.npy`.

## When to run
Run this notebook **once per dataset change**. After the artifacts are committed to the repo,
the live demo notebook (`harsh_week5_qwen3vl.ipynb`) loads them directly — this notebook does not
need to run again unless the CSV changes.

## Workflow: keeping Colab in sync with local changes

1. Edit code locally → run `./sync.sh` → pushes to GitHub
2. In Colab: **run the SYNC cell** (fast git pull, no reinstall) to pick up the latest code
3. On a fresh runtime: run **Cell 1 (Environment Setup)** once, restart, then continue from Stage 2

> **Note:** Cell 1 clones the repo from GitHub directly — no zip upload required.
> Google Drive is still mounted for the `uv` package cache and output artifacts.

## Stage Map
| Stage | Cell | Description |
|-------|------|-------------|
| SYNC | SYNC cell | Pull latest code from GitHub after `./sync.sh` — no reinstall |
| 1 | Cell 1 | Environment setup — git clone, install deps. **Restart runtime after this cell.** |
| 2 | Cell 2 | Config — paths, Drive mount for outputs, HF token |
| 3 | Cell 3 | Load & inspect dataset, resume from checkpoint if available |
| 4 | Cell 4 | LLM enrichment (Qwen3-8B, vLLM native guided decoding, batch 16) |
| 5 | Cell 5 | Corpus re-embedding (Qwen3-VL-Embedding-8B, batch 64) |
| 6 | Cell 6 | Verify artifacts |
| 7 | Cell 7 | Next steps — copy outputs to repo, update settings |


In [ ]:
# SYNC — Force-pull latest code from GitHub with verification
# Pulls code, verifies fix is present, and reports status.
import subprocess, os

REPO_DIR = '/content/vibescent'

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('[OK] Repo not yet cloned — skip this cell on first run. Run Cell 1 (Environment Setup) first.')
else:
    # Force-fetch all branches to ensure we see the latest commits
    print('SYNC: Force-fetching latest from GitHub...')
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'fetch', 'origin', 'main', '--force'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print('[ERROR] git fetch failed:')
        print(result.stderr.strip())
        raise RuntimeError('Could not fetch from GitHub')
    
    # Forcefully reset to origin/main (discards local edits, ensures exact GitHub version)
    print('SYNC: Resetting to origin/main...')
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print('[ERROR] git reset failed:')
        print(result.stderr.strip())
        raise RuntimeError('Could not reset to origin/main')
    
    # Verify the hardening fix is present by checking for the _pip_install function
    print('SYNC: Verifying notebook hardening fix...')
    with open(os.path.join(REPO_DIR, 'notebooks/harsh_offline_pipeline.ipynb'), 'r') as f:
        notebook_content = f.read()
    
    if '_pip_install' in notebook_content and 'globals().get(' in notebook_content:
        print('[OK] ✓ Hardening fix detected: _pip_install helper + defensive globals found.')
    else:
        print('[WARN] ✗ Hardening fix NOT found in notebook. GitHub may not be updated yet.')
        print('       Check https://github.com/GavinGongTech/VibeScent/commits/main')
    
    # Show current commit
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
        capture_output=True, text=True,
    )
    commit = result.stdout.strip() if result.returncode == 0 else 'unknown'
    print(f'\nSYNC complete. Current commit: {commit}')


In [ ]:
# Stage 1: Environment Setup
# Run once per new runtime, then restart runtime, then continue from Stage 2.
import subprocess, sys, traceback, os
import IPython

def custom_exc(shell, etype, evalue, tb, tb_offset=None):
    print("\n" + "="*60)
    print("!!! AN ERROR OCCURRED !!!")
    print("="*60)
    traceback.print_exception(etype, evalue, tb)
    print("="*60)
    print("!!! CHECK THE TRACEBACK ABOVE TO FIND THE EXACT LINE OF CODE !!!\n")

_ipython = IPython.get_ipython()
if _ipython:
    _ipython.set_custom_exc((Exception,), custom_exc)
    _ipython.magic("xmode Verbose")

REPO_DIR = '/content/vibescent'
GITHUB_REPO = 'https://github.com/GavinGongTech/VibeScent.git'

# Step 1: Clone or update repo from GitHub
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print("Step 1: Repo already present - pulling latest from GitHub...")
    result = subprocess.run(
        ['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
        capture_output=True, text=True,
    )
    print(result.stdout.strip() or 'Already up to date.')
else:
    print("Step 1: Cloning from GitHub (first run)...")
    subprocess.check_call(['git', 'clone', '--depth=1', GITHUB_REPO, REPO_DIR])
    print(f"Cloned to {REPO_DIR}")

os.chdir(REPO_DIR)

# Step 2: Mount Drive for pip cache + output artifacts
print("Step 2: Mounting Google Drive for package cache and outputs...")
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    PIP_CACHE = '/content/drive/MyDrive/pip_cache'
    os.makedirs(PIP_CACHE, exist_ok=True)
    os.environ['PIP_CACHE_DIR'] = PIP_CACHE
    print(f"pip cache: {PIP_CACHE}")
except Exception as e:
    print(f"[WARN] Drive mount skipped ({e}). Package cache will be ephemeral.")

# Step 3: Upgrade pip tooling
print("Step 3: Upgrading pip tooling...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'],
    check=True,
)


def _pip_install(packages, *, required=True, extra_args=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if extra_args:
        cmd.extend(extra_args)
    cmd.extend(packages)

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        return True

    details = (result.stderr or result.stdout or '').strip()
    if required:
        raise RuntimeError(
            f"pip install failed: {' '.join(packages)}\n{details[-2000:]}"
        )

    print(f"[WARN] Optional install failed: {' '.join(packages)}")
    if details:
        print(details.splitlines()[-1])
    return False


# Step 4a: Ensure torch is available before model installs
try:
    import torch  # noqa: F401
    print('Step 4a: torch already available in runtime.')
except Exception:
    print('Step 4a: Installing torch>=2.4,<3.0 ...')
    _pip_install(['torch>=2.4,<3.0'], required=True)

# Step 4b: Install required runtime dependencies
_required_pkgs = [
    'google-genai',
    'pandas',
    'numpy',
    'accelerate',
    'qwen-vl-utils>=0.0.14',
    'json-repair',
    'tqdm',
    'outlines',
    'transformers>=4.57.0,<5.0',
]
print("Step 4b: Installing required runtime dependencies...")
_pip_install(_required_pkgs, required=True)

# Step 4c: Install vibescent project in editable mode without re-resolving deps
print("Step 4c: Installing vibescent project (editable, no-deps)...")
_pip_install([REPO_DIR], required=True, extra_args=['--no-deps', '-e'])

# Step 4d: Optional accelerators (best effort)
print("Step 4d: Installing optional accelerators (best effort)...")
for _pkg in ['bitsandbytes', 'hf_transfer', 'vllm']:
    _ok = _pip_install([_pkg], required=False)
    if _pkg == 'vllm' and not _ok:
        print('[WARN] Stage 4 will auto-fallback to outlines/transformers backend.')

try:
    import torch
    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU runtime'
    print(f"Torch ready: {torch.__version__} | backend: {_gpu}")
except Exception as e:
    print(f"[WARN] torch import check failed: {e}")

print('\nEnvironment ready. Restart runtime now, then continue from Stage 2.')

In [ ]:
# Stage 2: Config
import os, sys
REPO_DIR = '/content/vibescent'
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

# Google Drive — all outputs saved here (survives session restart)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/vibescent_offline'
os.makedirs(DRIVE_BASE, exist_ok=True)

# Paths
INPUT_CSV      = os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv')
ENRICHED_CSV   = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv')
CHECKPOINT_CSV = os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv.ckpt')
FAILURES_LOG   = os.path.join(DRIVE_BASE, 'enrichment_failures.jsonl')
EMBEDDINGS_DIR = os.path.join(DRIVE_BASE, 'qwen3vl_corpus')
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

# INPUT_CSV is gitignored — copy from Drive if not present in the cloned repo.
# First-time setup: upload vibescent_enriched.csv to one of the Drive locations listed below.
# After the pipeline completes, Stage 7 copies the enriched CSV back here automatically.
if not os.path.exists(INPUT_CSV):
    import shutil as _shutil
    _drive_fallbacks = [
        os.path.join(DRIVE_BASE, 'vibescent_enriched_full.csv'),  # enriched output from a prior run
        '/content/drive/MyDrive/vibescent_enriched.csv',           # manual upload location
    ]
    for _src in _drive_fallbacks:
        if os.path.exists(_src):
            os.makedirs(os.path.dirname(INPUT_CSV), exist_ok=True)
            _shutil.copy(_src, INPUT_CSV)
            print(f'Copied INPUT_CSV from Drive: {_src}')
            break
    else:
        print(f'[ERROR] INPUT_CSV not found: {INPUT_CSV}')
        print( '        For first-time setup, upload vibescent_enriched.csv to one of:')
        for _c in _drive_fallbacks:
            print(f'          {_c}')
        raise FileNotFoundError('Input CSV missing. See instructions above.')
else:
    _size_mb = os.path.getsize(INPUT_CSV) // (1024 * 1024)
    print(f'Input CSV found in repo: {INPUT_CSV}  ({_size_mb} MB)')

# HF model cache on Drive — prevents re-downloading on each new session
HF_CACHE = '/content/drive/MyDrive/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# HF token from Colab Secrets (sidebar → 🔑 Secrets → + New secret: HF_TOKEN)
try:
    from google.colab import userdata as _ud
    _hf_token = _ud.get('HF_TOKEN') or ''
except Exception:
    _hf_token = os.environ.get('HF_TOKEN', '')

if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab Secrets.')
else:
    print('[WARN] HF_TOKEN not set — add via Colab sidebar → Secrets → HF_TOKEN')
    print('       Model download will fail without it.')

print(f'Config ready. Drive base: {DRIVE_BASE}')

In [ ]:
# Stage 3: Load and Inspect Dataset
import os, sys, shutil
import pandas as pd, numpy as np

# Defensive path fallbacks - safe even if Stage 2 was not re-run this session
REPO_DIR = globals().get('REPO_DIR', '/content/vibescent')
DRIVE_BASE_DEF = '/content/drive/MyDrive/vibescent_offline'
INPUT_CSV = globals().get('INPUT_CSV', os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'))
CHECKPOINT_CSV = globals().get('CHECKPOINT_CSV', os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv.ckpt'))

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from vibescents.enrich import ENRICHMENT_COLUMNS

# If INPUT_CSV is missing in repo clone, recover from Drive fallbacks
if not os.path.exists(INPUT_CSV):
    _drive_fallbacks = [
        os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv'),
        '/content/drive/MyDrive/vibescent_enriched.csv',
    ]
    for _src in _drive_fallbacks:
        if os.path.exists(_src):
            os.makedirs(os.path.dirname(INPUT_CSV), exist_ok=True)
            shutil.copy(_src, INPUT_CSV)
            print(f'Copied INPUT_CSV from Drive: {_src}')
            break
    else:
        raise FileNotFoundError(
            f'Input CSV missing: {INPUT_CSV}. Upload vibescent_enriched.csv to Drive first.'
        )

df_raw = pd.read_csv(INPUT_CSV)
print(f'Loaded: {df_raw.shape}')

# Show how many rows actually need enrichment
for col in ENRICHMENT_COLUMNS:
    filled = df_raw[col].notna().sum() if col in df_raw.columns else 0
    pct = filled / len(df_raw) * 100
    print(f'  {col}: {filled}/{len(df_raw)} filled ({pct:.1f}%)')

# Auto-resume: if checkpoint exists, use it as starting point
if os.path.exists(CHECKPOINT_CSV):
    df_ckpt = pd.read_csv(CHECKPOINT_CSV)
    print(f'\nCheckpoint found: {CHECKPOINT_CSV} ({df_ckpt.shape})')
    # Merge checkpoint into df_raw (indices must align - both loaded from the same source)
    for col in ENRICHMENT_COLUMNS:
        if col in df_ckpt.columns:
            if col not in df_raw.columns:
                df_raw[col] = None
            mask = df_ckpt[col].notna()
            if mask.any():
                df_raw.loc[mask[mask].index, col] = df_ckpt.loc[mask, col]
    already_done = df_raw['vibe_sentence'].notna().sum() if 'vibe_sentence' in df_raw.columns else 0
    print(f'After merge: {already_done} rows already enriched')

df_work = df_raw.copy()
if 'fragrance_id' not in df_work.columns:
    df_work.insert(0, 'fragrance_id', df_work.index.astype(str))
print(f'\nReady to enrich. Shape: {df_work.shape}')

In [ ]:
# Stage 4: LLM Enrichment (Qwen3-8B - local GPU, vLLM native guided decoding)
# Uses vLLM native guided decoding when available, then falls back to outlines/transformers.
# Alternatives: model_name = "google/gemma-3-12b-it" or "google/gemma-3-27b-it"
import sys, os, json, traceback, shutil
import torch, gc
import pandas as pd
from tqdm.auto import tqdm

# Defensive path fallbacks - safe even if Stage 2 was not re-run this session
REPO_DIR = globals().get('REPO_DIR', '/content/vibescent')
DRIVE_BASE_DEF = '/content/drive/MyDrive/vibescent_offline'
INPUT_CSV = globals().get('INPUT_CSV', os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'))
CHECKPOINT_CSV = globals().get('CHECKPOINT_CSV', os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv.ckpt'))
ENRICHED_CSV = globals().get('ENRICHED_CSV', os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv'))
FAILURES_LOG = globals().get('FAILURES_LOG', os.path.join(DRIVE_BASE_DEF, 'enrichment_failures.jsonl'))

os.makedirs(os.path.dirname(CHECKPOINT_CSV), exist_ok=True)
os.makedirs(os.path.dirname(ENRICHED_CSV), exist_ok=True)

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
# Auto-reload source code changes from GitHub (no kernel restart needed)
import importlib, sys
if 'vibescents' in sys.modules:
    for mod in list(sys.modules.keys()):
        if mod.startswith('vibescents'):
            del sys.modules[mod]

from vibescents.enrich import (
    VLLMNativeEnrichmentClient,
    QwenOutlinesEnrichmentClient,
    build_retrieval_text,
    ENRICHMENT_COLUMNS,
    _build_prompt,
    _serialize_value,
)

# Reconstruct df_work if Stage 3 was skipped after a runtime restart
if 'df_work' not in globals():
    if not os.path.exists(INPUT_CSV):
        _drive_fallbacks = [
            os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv'),
            '/content/drive/MyDrive/vibescent_enriched.csv',
        ]
        for _src in _drive_fallbacks:
            if os.path.exists(_src):
                os.makedirs(os.path.dirname(INPUT_CSV), exist_ok=True)
                shutil.copy(_src, INPUT_CSV)
                print(f'Copied INPUT_CSV from Drive: {_src}')
                break
        else:
            raise FileNotFoundError(
                f'Input CSV missing: {INPUT_CSV}. Run Stage 2 or upload CSV to Drive first.'
            )

    df_raw = pd.read_csv(INPUT_CSV)

    if os.path.exists(CHECKPOINT_CSV):
        df_ckpt = pd.read_csv(CHECKPOINT_CSV)
        print(f'Checkpoint found: {CHECKPOINT_CSV} ({df_ckpt.shape})')
        for col in ENRICHMENT_COLUMNS:
            if col in df_ckpt.columns:
                if col not in df_raw.columns:
                    df_raw[col] = None
                mask = df_ckpt[col].notna()
                if mask.any():
                    df_raw.loc[mask[mask].index, col] = df_ckpt.loc[mask, col]

    df_work = df_raw.copy()
    if 'fragrance_id' not in df_work.columns:
        df_work.insert(0, 'fragrance_id', df_work.index.astype(str))
    print(f'Reconstructed df_work from disk: {df_work.shape}')

ENRICHMENT_MODEL = "Qwen/Qwen3-8B"  # change to any HF instruct model
CHECKPOINT_EVERY = 100
BATCH_SIZE = 16  # vLLM handles batching natively

gc.collect()
torch.cuda.empty_cache()

print(f"Loading enrichment model: {ENRICHMENT_MODEL}")
print(f"VRAM before: {torch.cuda.memory_allocated()/1e9:.1f} GB")

try:
    client = VLLMNativeEnrichmentClient(model_name=ENRICHMENT_MODEL)
    print("Using vLLM native guided decoding backend.")
except Exception as e:
    _reason = str(e).splitlines()[0] if str(e) else type(e).__name__
    print(f"[WARN] vLLM backend unavailable ({_reason})")
    print("[INFO] Falling back to outlines/transformers backend.")
    client = QwenOutlinesEnrichmentClient(model_name=ENRICHMENT_MODEL)

print(f"VRAM after model load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Identify rows needing enrichment (skip already-filled rows on resume)
_need_mask = (
    df_work["vibe_sentence"].isna()
    if "vibe_sentence" in df_work.columns
    else pd.Series([True] * len(df_work), index=df_work.index)
)
_need_idx = df_work[_need_mask].index.tolist()
print(f"Rows needing enrichment: {len(_need_idx)} / {len(df_work)}")

for col in ENRICHMENT_COLUMNS:
    if col not in df_work.columns:
        df_work[col] = None

_completed, _failed = 0, 0
_failures = []

with tqdm(total=len(_need_idx), desc="Enriching (Batched)") as pbar:
    for i in range(0, len(_need_idx), BATCH_SIZE):
        batch_idx = _need_idx[i:i + BATCH_SIZE]
        batch_prompts = [_build_prompt(df_work.loc[idx]) for idx in batch_idx]

        try:
            records = client.generate_batch(batch_prompts)
            for idx, record in zip(batch_idx, records):
                if record is None:
                    _failed += 1
                    _failures.append({"idx": idx, "error": "Failed to parse model output into EnrichmentSchemaV2."})
                else:
                    for col in ENRICHMENT_COLUMNS:
                        df_work.at[idx, col] = _serialize_value(getattr(record, col))
                    _completed += 1
        except Exception as e:
            error_msg = f"Batch failed: {str(e)}\n{traceback.format_exc()}"
            print(f"\n[ERROR] {error_msg}")
            for idx in batch_idx:
                _failed += 1
                _failures.append({"idx": idx, "error": error_msg})

        pbar.update(len(batch_idx))
        pbar.set_postfix(ok=_completed, fail=_failed)

        if (_completed + _failed) >= CHECKPOINT_EVERY and (_completed + _failed) % CHECKPOINT_EVERY < BATCH_SIZE:
            df_work.to_csv(CHECKPOINT_CSV, index=False)

# Final checkpoint + retrieval_text
df_work.to_csv(CHECKPOINT_CSV, index=False)
df_work = build_retrieval_text(df_work)
df_work.to_csv(ENRICHED_CSV, index=False)

if _failures:
    with open(FAILURES_LOG, "w") as fh:
        for r in _failures:
            fh.write(json.dumps(r) + "\n")

print(f"\nEnrichment done: {_completed} ok, {_failed} failed")
print(f"Saved: {ENRICHED_CSV}")
print("\nSample vibe_sentences:")
for _, row in df_work[df_work["vibe_sentence"].notna()].head(3).iterrows():
    print(f"  {row['name']}: {str(row['vibe_sentence'])[:100]}")

# Release model weights before Stage 5 loads the embedder
try:
    if hasattr(client, '_llm'):
        client._llm = None
    del client
    import gc as _gc
    _gc.collect()
    torch.cuda.empty_cache()
    print(f'VRAM after releasing enrichment model: {torch.cuda.memory_allocated()/1e9:.1f} GB')
except Exception as _e:
    print(f'[WARN] Could not release enrichment model client: {_e}')

In [ ]:
# Stage 5: Corpus Re-embedding (Qwen3-VL-Embedding-8B)
# ONE-TIME: produces embeddings in the same vector space as query-time embedding
# Run AFTER Stage 4 so retrieval_text includes vibe_sentence + enriched fields
import sys, os, time, torch, gc, json, glob
import numpy as np
import pandas as pd
from pathlib import Path

# Defensive path fallbacks - safe to run even if Stage 2 was not re-run this session
REPO_DIR       = globals().get('REPO_DIR', '/content/vibescent')
DRIVE_BASE_DEF = '/content/drive/MyDrive/vibescent_offline'
ENRICHED_CSV   = globals().get('ENRICHED_CSV', os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv'))
INPUT_CSV      = globals().get('INPUT_CSV', os.path.join(REPO_DIR, 'data', 'vibescent_enriched.csv'))
EMBEDDINGS_DIR = globals().get('EMBEDDINGS_DIR', os.path.join(DRIVE_BASE_DEF, 'qwen3vl_corpus'))

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

gc.collect()
torch.cuda.empty_cache()

# TF32 for faster matmuls on Blackwell/Ampere
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Load enriched CSV (from Stage 4 output on Drive, fall back to raw input CSV)
_csv = ENRICHED_CSV if os.path.exists(ENRICHED_CSV) else INPUT_CSV
df_embed = pd.read_csv(_csv)
print(f'Embedding {len(df_embed)} rows from: {_csv}')

# Build retrieval_text if missing or empty
if 'retrieval_text' not in df_embed.columns or df_embed['retrieval_text'].isna().all():
    from vibescents.enrich import build_retrieval_text
    df_embed = build_retrieval_text(df_embed)

texts = df_embed['retrieval_text'].fillna(df_embed.get('name', '')).tolist()
if not texts:
    raise ValueError(f'No rows available for embedding from {_csv}')
print(f'Sample retrieval_text: {texts[0][:120]}')

# Load embedder - bump batch size for corpus throughput
from vibescents.embeddings import Qwen3VLMultimodalEmbedder
from vibescents.settings import Settings

Qwen3VLMultimodalEmbedder._BATCH_SIZE = 64  # throughput-optimised for corpus
_s = Settings()  # no extra args needed - model name is in defaults
embedder = Qwen3VLMultimodalEmbedder(settings=_s, load_in_8bit=False)

print(f'VRAM after embedder load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

out_emb = os.path.join(EMBEDDINGS_DIR, 'embeddings.npy')
out_manifest = os.path.join(EMBEDDINGS_DIR, 'manifest.json')
CKPT_DIR = os.path.join(EMBEDDINGS_DIR, 'ckpts')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

# Resume logic
def get_checkpoints():
    return sorted(
        glob.glob(os.path.join(CKPT_DIR, 'embeddings_ckpt_*.npy')),
        key=lambda p: int(Path(p).stem.split('_')[-1]),
    )

ckpt_files = get_checkpoints()
already_embedded = sum(np.load(f).shape[0] for f in ckpt_files)

if already_embedded > len(texts):
    raise ValueError(
        f'Checkpoint row count ({already_embedded}) exceeds dataset size ({len(texts)}). '
        f'Delete {CKPT_DIR} and re-run Stage 5.'
    )

print(f'Found {len(ckpt_files)} checkpoints. Already embedded: {already_embedded} / {len(texts)}')

texts_to_embed = texts[already_embedded:]

if texts_to_embed:
    print(f'Resuming embedding for remaining {len(texts_to_embed)} rows...')
    t0 = time.perf_counter()

    CHUNK_SIZE = Qwen3VLMultimodalEmbedder._BATCH_SIZE * 50  # checkpoint every 3200 rows

    from tqdm.auto import tqdm
    with tqdm(total=len(texts_to_embed), desc='Embedding (Outer Chunks)') as outer_pbar:
        for i in range(0, len(texts_to_embed), CHUNK_SIZE):
            chunk = texts_to_embed[i:i + CHUNK_SIZE]
            chunk_emb = embedder.embed_multimodal_documents(chunk)

            ckpt_path = os.path.join(CKPT_DIR, f'embeddings_ckpt_{len(get_checkpoints())}.npy')
            np.save(ckpt_path, chunk_emb)
            outer_pbar.update(len(chunk))

    elapsed = time.perf_counter() - t0
    print(f'Embedded {len(texts_to_embed)} rows in {elapsed:.0f}s  ({len(texts_to_embed)/elapsed:.0f} rows/s)')

# Consolidate all checkpoint shards
all_ckpts = get_checkpoints()
final_parts = [np.load(f) for f in all_ckpts]

if final_parts:
    embeddings = np.vstack(final_parts)
else:
    embeddings = np.empty((0, 0), dtype=np.float32)

print(f'Embeddings shape: {embeddings.shape}  dtype: {embeddings.dtype}')
assert embeddings.shape[0] == len(texts),     f'Expected {len(texts)} embeddings, got {embeddings.shape[0]}. Re-run from the last checkpoint.'

# Verify L2 normalization - embedder normalizes internally; assert here as a safety check
norms = np.linalg.norm(embeddings, axis=1)
print(f'Norms - mean: {norms.mean():.4f}  std: {norms.std():.5f}  min: {norms.min():.4f}')
print(f'NaN count: {np.isnan(embeddings).sum()}')
assert np.isnan(embeddings).sum() == 0, 'NaN values found in embeddings!'
assert norms.mean() > 0.99, f'Embeddings do not appear L2-normalized (mean norm={norms.mean():.4f}). Check the embedder.'

# Save
np.save(out_emb, embeddings)
manifest = {
    'model': 'Qwen/Qwen3-VL-Embedding-8B',
    'created': __import__('datetime').datetime.utcnow().isoformat(),
    'n_rows': int(embeddings.shape[0]),
    'dim': int(embeddings.shape[1]),
    'dtype': str(embeddings.dtype),
    'l2_normalized': True,
    'source_csv': _csv,
}
with open(out_manifest, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\nSaved embeddings: {out_emb}')
print(f'Saved manifest:   {out_manifest}')
print(json.dumps(manifest, indent=2))

In [ ]:
# Stage 6: Verify Artifacts
import numpy as np, json, os, pandas as pd

# Defensive path fallbacks
DRIVE_BASE_DEF = '/content/drive/MyDrive/vibescent_offline'
EMBEDDINGS_DIR = globals().get('EMBEDDINGS_DIR', os.path.join(DRIVE_BASE_DEF, 'qwen3vl_corpus'))
INPUT_CSV = globals().get('INPUT_CSV', '/content/vibescent/data/vibescent_enriched.csv')
ENRICHED_CSV = globals().get('ENRICHED_CSV', os.path.join(DRIVE_BASE_DEF, 'vibescent_enriched_full.csv'))

# Load df_embed if Stage 5 was skipped
if 'df_embed' not in dir():
    _csv = ENRICHED_CSV if os.path.exists(ENRICHED_CSV) else INPUT_CSV
    df_embed = pd.read_csv(_csv)
    print(f'Loaded df_embed from: {_csv}  shape={df_embed.shape}')

# Load and verify embeddings
emb = np.load(os.path.join(EMBEDDINGS_DIR, 'embeddings.npy'))
with open(os.path.join(EMBEDDINGS_DIR, 'manifest.json')) as f:
    mf = json.load(f)

print(f'Embeddings shape: {emb.shape}')
print(f'NaN count: {np.isnan(emb).sum()}')
assert np.isnan(emb).sum() == 0, 'NaN values in saved embeddings!'
assert emb.shape[0] > 0, 'Embeddings array is empty. Re-run Stage 5.'

norms = np.linalg.norm(emb, axis=1)
print(f'Norms - mean: {norms.mean():.4f}  std: {norms.std():.5f}')
assert norms.mean() > 0.99, f'Embeddings not L2-normalized (mean={norms.mean():.4f})'

assert emb.shape[0] == len(df_embed),     f'Shape mismatch: {emb.shape[0]} embeddings vs {len(df_embed)} DataFrame rows'

print(f'Manifest: {json.dumps(mf, indent=2)}')

# Sanity: top-5 most similar to row 0 (embeddings are L2-normalized, so dot = cosine sim)
if emb.shape[0] > 1:
    sims = emb @ emb[0]
    top5 = np.argsort(-sims)[1:6]
    name0 = df_embed.iloc[0].get('name', 'row 0')
    print(f'\nTop-5 most similar to "{name0}":')
    for i in top5:
        name_i = df_embed.iloc[i].get('name', f'row {i}')
        print(f'  [{i}] {name_i} - sim={sims[i]:.4f}')
else:
    print('Only one embedding row present; skipping similarity preview.')

print('\n=== All artifacts verified. Ready to commit to repo. ===')
print(f'Next step: copy {EMBEDDINGS_DIR}/embeddings.npy to artifacts/qwen3vl_corpus/ in the repo.')

# Stage 7: Next Steps — Copy Outputs to Repo

After all stages complete, copy the artifacts from Google Drive into the repo and commit:

```bash
# From repo root (or paste into a Colab shell cell with !)
mkdir -p artifacts/qwen3vl_corpus
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/embeddings.npy artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/qwen3vl_corpus/manifest.json  artifacts/qwen3vl_corpus/
cp /content/drive/MyDrive/vibescent_offline/vibescent_enriched_full.csv   data/vibescent_enriched.csv
```

> **First-time setup:** `data/vibescent_enriched.csv` is gitignored and not tracked in the repo.
> Before running Cell 2, upload your raw `vibescent_enriched.csv` to Google Drive at one of:
> - `MyDrive/vibescent_offline/vibescent_enriched.csv`
> - `MyDrive/vibescent_enriched.csv`
>
> Cell 2 will copy it into the Colab session automatically.
> After the pipeline finishes, the `cp` command above writes the enriched CSV back to the repo
> so future runs can read it directly from the clone.

## `settings.py` — Already Updated

`corpus_embeddings_path` in `src/vibescents/settings.py` already points to the correct location:

```python
corpus_embeddings_path: str = str(DEFAULT_ARTIFACTS_DIR / 'qwen3vl_corpus' / 'embeddings.npy')
```

No changes to `settings.py` are needed.

## Run the Week 5 Demo Notebook Next

Open `notebooks/harsh_week5_qwen3vl.ipynb`.
**Skip Stage 0** (corpus re-embedding) — it is now complete.
Continue from Stage 1 (setup) as normal.